# Phase 3 - KPI Calculations
**IndiaKart E-Commerce Analytics | Analyst: Bhavana**

All management KPIs from the brief, plus the bonus cohort retention analysis.
Results are exported to `../outputs/tables/kpis.json` and consumed by the Phase 4 dashboard.

In [1]:

import pandas as pd, numpy as np, json, os
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5.5)
plt.rcParams["axes.titlesize"] = 13

DATA = "../data"
CHARTS = "../outputs/charts"
TABLES = "../outputs/tables"
os.makedirs(CHARTS, exist_ok=True); os.makedirs(TABLES, exist_ok=True)

def rd(name, dates=()):
    df = pd.read_csv(f"{DATA}/{name}.csv")
    for c in dates:
        df[c] = pd.to_datetime(df[c], format="%d-%m-%Y", errors="coerce")
    return df

def inr(v):
    return f"Rs.{v:,.0f}"

orders     = rd("orders", ["order_date", "delivered_date"])
order_items= rd("order_items")
customers  = rd("customers", ["registration_date", "last_login_date"])
products   = rd("products", ["launch_date"])
payments   = rd("payments", ["payment_date", "refund_date"])
returns    = rd("returns", ["return_date", "refund_date"])
inventory  = rd("inventory", ["last_restocked_date"])
suppliers  = rd("suppliers", ["created_date"])
print("All 8 tables loaded")


All 8 tables loaded


## 3.1 Headline KPIs

In [2]:
total_orders = len(orders)
gmv = orders["final_amount"].sum()
delivered = orders[orders["status"] == "Delivered"]
net_revenue = delivered["final_amount"].sum()
aov = net_revenue / len(delivered)
cancel_rate = (orders["status"] == "Cancelled").mean() * 100
return_rate = len(returns) / len(delivered) * 100
pay_fail_rate = (payments["status"] == "Failed").mean() * 100
fill_rate = (inventory["status"] == "In Stock").mean() * 100
repeat_rate = (customers["total_orders"] > 1).mean() * 100

kpis = {
 "total_orders": int(total_orders),
 "gmv": float(gmv),
 "net_revenue": float(net_revenue),
 "aov": float(aov),
 "delivered_orders": int(len(delivered)),
 "cancellation_rate": float(cancel_rate),
 "return_rate": float(return_rate),
 "payment_failure_rate": float(pay_fail_rate),
 "inventory_fill_rate": float(fill_rate),
 "repeat_purchase_rate": float(repeat_rate),
 "total_customers": int(len(customers)),
 "total_gst": float(orders["gst_amount"].sum()),
}
for k, v in kpis.items():
    print(f"{k:24s}: {v:,.2f}")

total_orders            : 50,000.00
gmv                     : 3,147,385,544.58
net_revenue             : 2,053,858,611.96
aov                     : 63,197.59
delivered_orders        : 32,499.00
cancellation_rate       : 11.99
return_rate             : 30.77
payment_failure_rate    : 3.50
inventory_fill_rate     : 92.20
repeat_purchase_rate    : 88.38
total_customers         : 10,000.00
total_gst               : 466,958,188.56


**Read-out vs. targets:** cancellation rate target is under 10%, return rate under 8%,
payment failure under 2% and inventory fill above 95%. The printed values above show where
IndiaKart stands against each.

## 3.2 Customer Lifetime Value by segment

In [3]:
clv = customers.groupby("segment").agg(
    customers=("customer_id", "count"),
    avg_total_spent=("total_spent", "mean"),
    avg_orders=("total_orders", "mean")).round(2).sort_values("avg_total_spent", ascending=False)
clv.to_csv(f"{TABLES}/clv_by_segment.csv")
clv

,customers,avg_total_spent,avg_orders
segment,,,
Premium,1501,532735.00,8.61
Regular,4026,361870.72,5.73
New,1488,274866.48,4.34
Budget,2505,186178.81,2.90
Inactive,480,32249.08,0.56


## 3.3 Category concentration and category KPIs

In [4]:
cat = order_items.groupby("category").agg(
    revenue=("total_price", "sum"), units=("quantity", "sum")).sort_values("revenue", ascending=False)
cat["revenue_share_pct"] = (cat["revenue"]/cat["revenue"].sum()*100).round(2)

ret_orders = returns.merge(order_items[["order_id", "category"]].drop_duplicates("order_id"),
                           on="order_id", how="left")
cat_returns = ret_orders["category"].value_counts()
cat_orders = order_items.drop_duplicates("order_id")["category"].value_counts()
cat["return_rate_pct"] = (cat_returns/cat_orders*100).round(2)
cat.to_csv(f"{TABLES}/category_kpis.csv")
print("Top category share: %.2f%% (concentration risk if > 60%%)" % cat["revenue_share_pct"].max())
cat

Top category share: 57.41% (concentration risk if > 60%)


,revenue,units,revenue_share_pct,return_rate_pct
category,,,,
Electronics,1.642423e+09,20309,57.41,19.63
Sports & Fitness,4.972384e+08,19816,17.38,21.53
Home & Kitchen,2.442952e+08,20161,8.54,20.88
Fashion,1.481553e+08,20103,5.18,20.91
Automotive,1.234851e+08,20026,4.32,20.51
Office Supplies,9.302935e+07,20292,3.25,20.22
Toys & Baby,4.658385e+07,19961,1.63,22.41
Beauty & Health,3.301811e+07,19635,1.15,19.34
Grocery,1.871868e+07,20099,0.65,19.91


## 3.4 Monthly KPI trend

In [5]:
orders["order_month"] = orders["order_date"].dt.to_period("M").astype(str)
monthly = orders.groupby("order_month").agg(
    orders=("order_id", "count"),
    gmv=("final_amount", "sum"),
    cancelled=("status", lambda s: (s == "Cancelled").sum())).reset_index()
delivered_m = delivered.assign(order_month=delivered["order_date"].dt.to_period("M").astype(str)) \
    .groupby("order_month").agg(net_revenue=("final_amount", "sum"),
                                delivered_orders=("order_id", "count")).reset_index()
monthly = monthly.merge(delivered_m, on="order_month", how="left")
monthly["aov"] = (monthly["net_revenue"]/monthly["delivered_orders"]).round(2)
monthly["cancellation_rate_pct"] = (monthly["cancelled"]/monthly["orders"]*100).round(2)
monthly.to_csv(f"{TABLES}/monthly_kpis.csv", index=False)
monthly.head(12)

,order_month,orders,gmv,cancelled,net_revenue,delivered_orders,aov,cancellation_rate_pct
0,2023-06,444,2.738485e+07,55,18012213.92,277,65026.04,12.39
1,2023-07,2081,1.289205e+08,232,85221985.85,1342,63503.72,11.15
2,2023-08,2091,1.281220e+08,254,82388045.94,1349,61073.42,12.15
3,2023-09,2104,1.374625e+08,254,89461680.62,1368,65395.97,12.07
4,2023-10,2060,1.324709e+08,240,85246602.95,1355,62912.62,11.65
5,2023-11,2114,1.372634e+08,235,84790991.85,1381,61398.26,11.12
6,2023-12,2090,1.369234e+08,254,91521374.89,1348,67894.20,12.15
7,2024-01,2090,1.314500e+08,258,86430666.32,1374,62904.42,12.34
8,2024-02,1907,1.209459e+08,229,75031911.46,1216,61703.87,12.01
9,2024-03,2111,1.336701e+08,249,86057363.50,1385,62135.28,11.80


## 3.5 Top products, states and operations tables (feed the dashboard)

In [6]:
top_products = order_items.groupby(["product_id", "product_name", "category"]).agg(
    revenue=("total_price", "sum"), units=("quantity", "sum")).reset_index() \
    .sort_values("revenue", ascending=False).head(10)
top_products.to_csv(f"{TABLES}/top_products.csv", index=False)

state_perf = orders.groupby("state").agg(orders=("order_id", "count"),
                                         gmv=("final_amount", "sum")).sort_values("orders", ascending=False).head(10)
state_perf.to_csv(f"{TABLES}/state_performance.csv")

new_cust = customers.groupby(customers["registration_date"].dt.to_period("M").astype(str)) \
    .size().rename("new_customers")
new_cust.to_csv(f"{TABLES}/new_customers_monthly.csv")

pay_break = payments.groupby(["payment_method", "status"]).size().unstack(fill_value=0)
pay_break.to_csv(f"{TABLES}/payment_breakdown.csv")

inv_status = inventory["status"].value_counts()
low_stock = inventory[inventory["status"].isin(["Low Stock", "Out of Stock"])] \
    .merge(products[["product_id", "product_name", "category"]], on="product_id", how="left") \
    .sort_values("quantity_available")[["product_id", "product_name", "category",
                                        "warehouse_location", "quantity_available",
                                        "reorder_level", "status"]].head(20)
low_stock.to_csv(f"{TABLES}/low_stock_products.csv", index=False)
warehouse = inventory.groupby("warehouse_location")["total_inventory_value"].sum() \
    .sort_values(ascending=False)
warehouse.to_csv(f"{TABLES}/warehouse_value.csv")
inv_status.to_csv(f"{TABLES}/inventory_status.csv")
returns["reason"].value_counts().to_csv(f"{TABLES}/return_reasons.csv")
orders["status"].value_counts().to_csv(f"{TABLES}/order_status.csv")
customers["segment"].value_counts().to_csv(f"{TABLES}/customer_segments.csv")
top_products

,product_id,product_name,category,revenue,units
66,PRD0067,Ambrane Power Bank Classic,Electronics,41923593.45,266
85,PRD0086,Godrej Camera Elite,Electronics,41223343.23,268
45,PRD0046,Pidilite Headphones Premium,Electronics,36386592.40,225
72,PRD0073,Maruti Smartwatch Plus,Electronics,33065784.55,206
76,PRD0077,Mivi Smartwatch Plus,Electronics,32515799.09,220
61,PRD0062,Godrej Power Bank Standard,Electronics,32398023.45,238
98,PRD0099,Bajaj Router Classic,Electronics,31995407.72,214
17,PRD0018,Infosys Laptop Ultra,Electronics,31652187.05,210
53,PRD0054,Zebronics Bluetooth Speaker Standard,Electronics,31607690.08,250
2,PRD0003,Fire-Boltt Smartphone Max,Electronics,30954801.13,240


## 3.6 Bonus - cohort retention analysis

In [7]:
co = orders.merge(customers[["customer_id", "registration_date"]], on="customer_id", how="left")
co = co.dropna(subset=["registration_date"])
co["cohort"] = co["registration_date"].dt.to_period("M")
co["order_period"] = co["order_date"].dt.to_period("M")
co["month_index"] = (co["order_period"] - co["cohort"]).apply(lambda x: x.n)
co = co[co["month_index"].between(0, 11)]
cohort = co.pivot_table(index="cohort", columns="month_index",
                        values="order_id", aggfunc="count").fillna(0).astype(int)
cohort.to_csv(f"{TABLES}/cohort_orders.csv")

fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(cohort, cmap="YlGnBu", annot=False, ax=ax)
ax.set_title("Cohort analysis - orders placed by registration cohort, by months since signup")
ax.set_xlabel("Months since registration"); ax.set_ylabel("Registration cohort")
fig.tight_layout(); fig.savefig(f"{CHARTS}/11_cohort_heatmap.png", dpi=140); plt.show()
cohort.head(8)

month_index,0,1,2,3,4,5,6,7,8,9,10,11
cohort,,,,,,,,,,,,
2022-07,0,0,0,0,0,0,0,0,0,0,0,9
2022-08,0,0,0,0,0,0,0,0,0,0,4,37
2022-09,0,0,0,0,0,0,0,0,0,9,35,41
2022-10,0,0,0,0,0,0,0,0,3,29,27,30
2022-11,0,0,0,0,0,0,0,3,29,43,25,35
2022-12,0,0,0,0,0,0,5,29,27,34,37,33
2023-01,0,0,0,0,0,6,27,29,33,41,34,40
2023-02,0,0,0,0,5,33,36,49,32,35,43,41


**Observation:** Activity is highest in the first months after registration and then settles at a
lower steady level, which is the normal shape for a marketplace. Retention programmes should
target months 2-4, where the drop-off is steepest.

## 3.7 Export KPIs for the dashboard and report

In [8]:
kpis["top_category"] = cat.index[0]
kpis["top_category_share"] = float(cat["revenue_share_pct"].max())
kpis["top_state"] = state_perf.index[0]
kpis["out_of_stock_skus"] = int((inventory["status"] == "Out of Stock").sum())
kpis["low_stock_skus"] = int((inventory["status"] == "Low Stock").sum())
kpis["top_return_reason"] = returns["reason"].value_counts().index[0]
kpis["premium_aov"] = float(delivered.merge(customers[["customer_id", "segment"]], on="customer_id")
                            .groupby("segment")["final_amount"].mean().max())
kpis["date_from"] = str(orders["order_date"].min().date())
kpis["date_to"] = str(orders["order_date"].max().date())
json.dump(kpis, open(f"{TABLES}/kpis.json", "w"), indent=2)
print(json.dumps(kpis, indent=2))

{
  "total_orders": 50000,
  "gmv": 3147385544.58,
  "net_revenue": 2053858611.9599998,
  "aov": 63197.59414012738,
  "delivered_orders": 32499,
  "cancellation_rate": 11.991999999999999,
  "return_rate": 30.770177543924426,
  "payment_failure_rate": 3.496,
  "inventory_fill_rate": 92.2,
  "repeat_purchase_rate": 88.38000000000001,
  "total_customers": 10000,
  "total_gst": 466958188.55999994,
  "top_category": "Electronics",
  "top_category_share": 57.41,
  "top_state": "Uttar Pradesh",
  "out_of_stock_skus": 2,
  "low_stock_skus": 76,
  "top_return_reason": "Size Issue",
  "premium_aov": 64951.14159389572,
  "date_from": "2023-06-24",
  "date_to": "2025-06-23"
}
